In [ ]:
!pip -q install -U vllm

In [ ]:
!pip -q install -U transformers huggingface_hub

In [ ]:
# A100 에서 돌릴때만 실행
!pip install -U "protobuf>=5.26.1,<6"

In [ ]:
import os

os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["HF_TOKEN"] = "..."
print(os.environ.get("HF_TOKEN"))

from vllm import LLM, SamplingParams

model_id = 'Qwen/Qwen3-32B-AWQ'
#model_id = 'Qwen/Qwen3-32B'
hf_token = os.environ["HF_TOKEN"]

llm = LLM(
    model=model_id,
    hf_token=hf_token,
    trust_remote_code=True,
    # 양자화 안할때만
    #dtype="float16",
    #max_model_len=8196,
)

#토큰 확인용
#from transformers import AutoTokenizer
#qwen_tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-32B", trust_remote_code=True)

In [ ]:
import pandas as pd

tmp = pd.read_csv('./tmp3.csv')

In [ ]:
'''프롬프트 구성'''

import ast
import json
import numpy as np
import pandas as pd

def clip_text(x, n=200):
    if not isinstance(x, str):
        return ""
    x = x.strip()
    return x if len(x) <= n else x[:n] + "..."

# pandas/numpy 자료형 -> 파이썬 기본 타입 변환 (json용)
def to_py(x):
    # pandas/numpy NaN 처리
    if x is None:
        return None
    if isinstance(x, float) and (x != x):  # NaN
        return None

    # numpy scalar -> python scalar
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, (np.bool_,)):
        return bool(x)


    # dict/list 재귀 처리
    if isinstance(x, dict):
        return {k: to_py(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [to_py(v) for v in x]

    return x

SYSTEM_PROMPT = (
    "너는 추천 시스템의 '추천 이유'를 설명하는 한국어 AI야. "
    "입력으로 제공된 JSON 데이터만 사용해서 특정 회사에 특정 과제가 왜 추천되었는지 한국어로 설명해."
    "[내부 사고 단계 - 출력 금지]\n"
    "1) 회사 목적/관심사를 이해한다.\n"
    "2) 과제의 핵심 기술/주요 내용을 파악한다.\n"
    "3) 회사 목적/관심사와 과제 내용의 연관성을 설명한다.\n"
    "4) sim_keyword가 비어있지 않다면 키워드 연관성을 설명한다."
    "이때 각 키워드가 회사 키워드인지 과제 키워드인지 명확히 구분하여 언급한다."
    "두 키워드가 공통적으로 수행하는 기능 또는 동작을 일반인이 이해할 수 있는 수준으로 풀어서 설명하고,"
    "해당 기능이 회사의 기술 관심사와 과제의 목적 또는 요구사항과 어떻게 연결되는지 서술한다."
    "추상적인 평가 표현만으로 설명을 끝내지 않는다.\n"
    "5) project_score 및 cosine_distance를 정량 근거로 해석한다.\n"
    "6) 논문/특허가 있으면 회사 및 과제와의 연관성을 중심으로 추천의 근거로 포함한다.\n"
    "7) 모든 요소를 섹션 구조로 일관되게 정리한다.\n\n"
    "[출력 규칙]\n"
    "- 위 내부 사고 단계나 중간 판단, 계산 과정은 절대 출력하지 않는다.\n"
    "- 추측하거나 없는 사실을 만들지 않는다.\n"
    "- 입력 JSON에 값이 없는 항목은 언급하지 않는다.\n"
    "- 모든 설명은 '평가'가 아니라 '추천 이유 설명'의 관점에서 작성한다.\n"
    "- 정량 지표(project_score, cosine_distance)는 각각 따로 평가하지 말고, 서로 보완적으로 해석하여 추천이 가능한 이유를 중심으로 설명한다.\n"
    )



FEWSHOT_MESSAGES = [
    {"role": "user", "content": (
        "아래 JSON만을 근거로 추천 근거를 작성해.\n"
        "출력은 JSON 하나만 반환해. (JSON 외 텍스트 금지)\n"
        "키는 output_requirements.format에 있는 섹션 제목을 그대로 사용해.\n\n"
        + json.dumps({
            "company": {
                "company_id": "C001",
                "name": "A사",
                "company_score": 90.12,
                "purpose": "반도체 공정용 소재의 제조 및 판매",
                "keywords": "박막, 코팅, 기판"
            },
            "project": {
                "project_id": "P001",
                "title": "금속 기판 기반 기능성 박막 증착 공정 개발",
                "keywords": "금속 기판, 증착, 코팅",
                "project_score": 78.34,
                "cosine_distance": 0.03
            },
            "keyword_links": {
                "sim_keyword": [
                    {"company": "코팅장치", "project": "코팅", "sim": 0.82},
                    {"company": "금속화합물막", "project": "금속 기판", "sim": 0.80}
                ]
            },
            "related_research": {
                "paper": "반응성 증착을 통한 박막 성장 및 계면 특성 분석",
                "patent": "금속 기판 상 기능성 박막 제조 방법 및 결함 제어 기술"
            },
            "output_requirements": {
                "language": "ko",
                "format": [
                    "추천 개요",
                    "회사 목적과 과제의 연관성",
                    "키워드 기반 기술적 연관성",
                    "정량 지표 기반 추천 근거",
                    "관련 논문 및 특허를 통한 기술 신뢰성",
                    "종합 추천 의견"
                ],
                "section_sentence_range": "각 섹션 3~4문장",
                "keyword_rule": "sim_keyword가 빈 리스트이면 '키워드 기반 기술적 연관성' 섹션을 아예 작성하지 않는다.",
                "numeric_rules": {
                    "decimal_places": 2,
                    "project_score_interpretation": "70점 평균 기준으로 높고 낮음을 함께 설명(높을수록 유리)",
                    "cosine_distance_interpretation": "코사인 거리 값이 작을수록 유사하다는 의미로 설명"
                },
                "paper_patent_rule": "paper와 patent는 값이 있을 때만 언급하고, 값이 없으면 '관련 논문 및 특허를 통한 기술 신뢰성' 섹션을 아예 작성하지 않는다.",
                "forbidden": ["설명", "예시", "참고", "제출", "메타"]
            }
        }, ensure_ascii=False)
    )},

    {"role": "assistant", "content": json.dumps({
        "추천 개요": (
            "A사는 반도체 공정용 소재 제조를 기반으로 박막·코팅·기판 관련 기술 수요가 있습니다. "
            "금속 기판 기반 기능성 박막 증착 공정 개발 과제는 금속 기판 위에 박막을 증착·코팅해 목표 성능을 확보하는 내용을 다룹니다. "
            "A사의 제조 공정 역량과 과제의 공정 중심 목표가 맞물려 연계 가능성이 큽니다."
        ),
        "회사 목적과 과제의 연관성": (
            "A사의 목적은 반도체 공정용 소재를 제조·판매하는 것으로, 기판 위 박막 형성 같은 정밀 공정 기술이 핵심입니다. "
            "금속 기판 기반 기능성 박막 증착 공정 개발 과제는 기판-박막 계면 품질과 공정 조건 제어를 통해 성능을 구현하려는 구조입니다. "
            "따라서 공정 최적화로 기능성 막 품질을 확보한다는 점에서 A사의 사업 목적과 과제 요구가 직접 연결됩니다."
        ),
        "키워드 기반 기술적 연관성": (
            "회사 키워드 ‘코팅장치’와 과제 키워드 ‘코팅’은 표면에 물질을 균일하게 입혀 성능을 부여하는 공정 동작을 공유합니다. "
            "이 공정의 재현성과 균일도 확보는 과제의 박막 품질 요구와 직결되며, A사의 장비·공정 운용 경험이 적용될 여지가 있습니다. "
            "또한 회사 키워드 ‘금속화합물막’과 과제 키워드 ‘금속 기판’은 금속 기반 위에 기능성 막을 형성해 물성을 제어한다는 흐름이 같아, 과제의 계면 안정화·결함 관리 이슈와 A사의 기술 관심사가 맞물립니다."
        ),
        "정량 지표 기반 추천 근거": (
            "과제의 project_score 78.34는 70점 기준 대비 높은 편으로, 적용 가능한 수준의 기술 구체성이 있음을 시사합니다. "
            "cosine_distance 0.03은 값이 작아 A사의 기술 맥락과 과제 주제가 매우 가깝다는 의미입니다. "
            "두 지표를 종합하면, 성숙도가 확보된 과제가 A사의 관심 영역과 높은 정합성을 보여 추천 타당성이 강화됩니다."
        ),
        "관련 논문 및 특허를 통한 기술 신뢰성": (
            "논문은 반응성 증착 기반 박막 성장과 계면 특성 분석을 다뤄 금속 기판 기반 박막 형성이라는 과제 핵심을 뒷받침합니다. "
            "특허는 금속 기판 상 박막 제조와 결함 제어 기술을 포함해 제조 관점의 구현 경로를 제시합니다. "
            "이는 A사의 박막·코팅 공정 역량과 결합될 때 공정 최적화 및 품질 안정화 근거로 활용될 수 있습니다."
        ),
        "종합 추천 의견": (
            "A사는 박막·코팅·기판 중심의 제조 역량을 보유하고, 해당 과제는 동일한 공정 축에서 성능 구현을 목표로 합니다. "
            "키워드 연결이 공정 동작 단위에서 성립하며, project_score와 cosine_distance가 기술 정합성을 함께 지지합니다. "
            "따라서 본 과제는 A사의 기존 역량을 활용해 확장 적용이 가능한 추천 대상으로 판단됩니다."
        )
    }, ensure_ascii=False)}
]



def build_company_single_prompt(company_row, rec_row):
    c_id = company_row["company_id"]
    c_name = company_row["company_name"]
    c_score = company_row.get("company_score", "")
    c_purpose = company_row.get("company_purpose", "")
    c_keyword = company_row.get("키워드_company", "")

    pid = rec_row["project_id"]
    pname = rec_row["project_name"]
    pscore = rec_row.get("project_score", "")
    dist = rec_row.get("cosine_distance", "")
    keyword_proj = rec_row.get("keyword_project", "")

    paper = rec_row.get("paper", "")
    patent = rec_row.get("patent", "")

    paper = "" if (paper is None or (hasattr(pd, "isna") and pd.isna(paper))) else clip_text(str(paper), 200)
    patent = "" if (patent is None or (hasattr(pd, "isna") and pd.isna(patent))) else clip_text(str(patent), 200)

    # keyword_links 파싱 및 필터
    sim_keyword_all = rec_row.get("keyword_links", []) or []
    if isinstance(sim_keyword_all, str):
        try:
            sim_keyword_all = ast.literal_eval(sim_keyword_all)
        except:
            sim_keyword_all = []

    sim_keyword = []
    for x in sim_keyword_all:
        if not isinstance(x, dict):
            continue
        try:
            if float(x.get("sim", -1)) >= 0.75:
                sim_keyword.append({
                    "company": x.get("company", ""),
                    "project": x.get("project", ""),
                    "sim": x.get("sim", None)
                })
        except:
            pass

    # 모델에 넘길 입력 (JSON형태)
    payload = {
        "company": {
            "company_id": c_id,
            "name": c_name,
            "company_score": c_score,
            "purpose": c_purpose,
            "keywords": c_keyword
        },
        "project": {
            "project_id": pid,
            "title": pname,
            "keywords": keyword_proj,
            "project_score": pscore,
            "cosine_distance": dist
        },
        "keyword_links": {
            "sim_keyword": sim_keyword  # 비어있을 수 있음
        },
        "related_research": {
            "paper": paper,
            "patent": patent
        },
        "output_requirements": {
            "language": "ko",
            "format": [
                "추천 개요",
                "회사 목적과 과제의 연관성",
                "키워드 기반 기술적 연관성",
                "정량 지표 기반 추천 근거",
                "관련 논문 및 특허를 통한 기술 신뢰성",
                "종합 추천 의견"
            ],
            "section_sentence_range": "각 섹션 3~4문장",
            "keyword_rule": "sim_keyword가 빈 리스트이면 '키워드 기반 기술적 연관성' 섹션을 아예 작성하지 않는다.",
            "numeric_rules": {
                "decimal_places": 2,
                "project_score_interpretation": "70점 평균 기준으로 높고 낮음을 함께 설명(높을수록 유리)",
                "cosine_distance_interpretation": "코사인 거리 값이 작을수록 유사하다는 의미로 설명"
            },
            "paper_patent_rule": "paper와 patent는 값이 있을 때만 언급하고, 값이 없으면 '관련 논문 및 특허를 통한 기술 신뢰성' 섹션을 아예 작성하지 않는다.",
            "forbidden": ["설명", "예시", "참고", "제출", "메타"]
        }
    }

    payload = to_py(payload)
    user_prompt = (
        "아래 JSON만을 근거로 추천 근거를 작성해.\n"
        "출력은 JSON 하나만 반환해. JSON 외 텍스트 금지야. \n"
        "첫 글자는 {, 마지막 글자는 } 로 끝내.\n"
        "``` 같은 코드블록은 절대 쓰지 마.\n"
        "키는 output_requirements.format에 있는 섹션 제목을 그대로 사용해.\n\n"
        f"{json.dumps(payload, ensure_ascii=False)}"
    )

    return [{"role": "system", "content": SYSTEM_PROMPT}] + FEWSHOT_MESSAGES + [{"role": "user", "content": user_prompt}]

In [ ]:
'''LLM 모델 호출 후 설명 생성 함수'''

import torch, gc
import re
from vllm import SamplingParams


def _messages_to_prompt(messages):
    """
    vLLM 프롬프트 문자열로 변환.
    - assistant 시작 토큰을 넣어줘야 모델이 답을 출력하기 시작함.
    """
    parts = []
    for m in messages:
        role = m["role"]
        content = m["content"].strip()
        if role == "system":
            parts.append(f"<|im_start|system\n{content}<|im_end|>")
        elif role == "user":
            parts.append(f"<|im_start|user\n{content}<|im_end|>")
        elif role == "assistant":
            parts.append(f"<|im_start|assistant\n{content}<|im_end|>")

    # 답변 시작 지점
    parts.append("<|im_start|assistant\n")
    return "\n".join(parts)

@torch.inference_mode()
def generate_explanation(messages, tokenizer, model,
                         max_new_tokens=1024, temperature=0.3, top_p=0.9):


    prompt = _messages_to_prompt(messages)

    params = SamplingParams(
        temperature=0.3,
        top_p=0.9,
        max_tokens=1024,
        # 채팅 끝
        stop=["<|im_end|>"]
    )

    outputs = llm.generate([prompt], params)
    result = outputs[0].outputs[0].text.strip()
    #print('check:::: ', result[:1000])

    # think 태그 제거 (모델이 출력할 경우 대비)
    result = re.sub(r"<think>.*?</think>\s*", "", result, flags=re.DOTALL).strip()
    result = re.sub(r"^\s*</think>\s*", "", result).strip()
    #print('check22:::: ', result[:1000])
    return result

In [ ]:
'''설명 생성 실행'''

import json
from itertools import islice
import re
import os
import time
import pandas as pd

# 입력 토큰 수 확인
def count_prompt_tokens(messages, tok):
    prompt = _messages_to_prompt(messages)
    ids = tok(prompt, add_special_tokens=False).input_ids
    return len(ids)

def hanja(text):
    return bool(re.search(r'[\u4e00-\u9fff]', text))

out_path = './qwen_model3_fewshot_awq_resultJson.csv'
eval_path = './qwen_model3_fewshot_awq_resultJson_evalLog.csv'

# 기존 파일 삭제
if os.path.exists(out_path):
    os.remove(out_path)
file_exists = os.path.exists(out_path)  # False

if os.path.exists(eval_path):
    os.remove(eval_path)
file_exists_eval = os.path.exists(eval_path)  # False

start_time = time.time()

for company_id, block in tmp.groupby("company_id", sort=False):
    company_time_start = time.time()

    eval_rows = []
    company_name = block["company_name"].iloc[0]
    company_row = block.iloc[0]
    explanations = []
    company_rows = []

    for _, rec_row in block.iterrows():
        messages = build_company_single_prompt(company_row, rec_row)

        '''
        # 입력 토큰 수 확인용
        prompt_tokens = count_prompt_tokens(messages, qwen_tok)
        print("prompt_tokens =", prompt_tokens)
        '''

        one = generate_explanation(messages, tokenizer=None, model=None)
        explanations.append(one)
        #print('one: ', one)

        # JSON 원샷 파싱 성공 여부
        valid_json = 1
        parsed_json = ""
        try:
            obj = json.loads(one)
            parsed_json = json.dumps(obj, ensure_ascii=False)
        except Exception:
            valid_json = 0

        # 평가용 로그 저장
        eval_rows.append({
            "company_id": company_id,
            "company_name": company_name,
            "project_id": rec_row.get("project_id", ""),
            "project_name": rec_row.get("project_name", ""),
            "valid_json": valid_json,
            "raw_output": one,          # 모델이 실제로 출력한 원문
            "parsed_json": parsed_json  # 파싱 성공 시 정규화된 JSON
        })

        company_rows.append({
            "company_id": company_id,
            "company_name": company_name,
            "explanation": one
        })

    company_time = time.time() - company_time_start
    print(f"===========company_name: {company_name} // company_id: {company_id} // time: {time.strftime('%H:%M:%S', time.gmtime(company_time))}==========")

    # 평가용 로그 저장 파일
    pd.DataFrame(eval_rows).to_csv(
        eval_path,
        mode="a",
        header=not file_exists_eval,
        index=False,
        encoding="utf-8-sig"
    )
    file_exists_eval = True

    explain_df = pd.DataFrame(company_rows)
    explain_df.to_csv(
        out_path,
        mode="a",
        header=not file_exists,
        index=False,
        encoding='utf-8-sig'
    )
    file_exists = True

    print('explain_df: ', explain_df.head())